### Restart and Run All Cells

In [1]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
year = 2026
quarter = 3
current_time = datetime.now()
formatted_time = current_time.strftime("%Y-%m-%d %H:%M:%S")
print(formatted_time )

2026-08-22 15:57:53


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',   
                'latest_amt':'{:,}','previous_amt':'{:,}','inc_amt':'{:,}',
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%','inc_pct':'{:.2f}%'
              }

In [5]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC
"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 3) 
OR (year = 2026-1 AND quarter >= 3+1)
ORDER BY year DESC, quarter DESC



In [7]:
dfc = pd.read_sql(sql, conlt)
dfc["Counter"] = 1
dfc_grp = dfc.groupby(["name"], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp["Counter"] == 4]
dfc_grp.shape

(5, 5)

In [9]:
sql = """
SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = %s AND quarter <= %s-1) 
OR (year = %s-1 AND quarter >= %s) 
ORDER BY year DESC, quarter DESC"""
sql = sql % (year, quarter, year, quarter)
print(sql)


SELECT name,year,quarter,q_amt 
FROM epss 
WHERE (year = 2026 AND quarter <= 3-1) 
OR (year = 2026-1 AND quarter >= 3) 
ORDER BY year DESC, quarter DESC


In [11]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.sample(5).style.format(format_dict)

,name,year,quarter,q_amt,Counter
66,GPSC,8102,10,"6,778,168",4
47,CPF,8102,10,"15,222,529",4
182,TTLPF,8102,10,"253,884",4
17,BA,8102,10,"3,894,524",4
85,KCE,8102,10,"930,922",4


In [13]:
dfp.name.unique().shape

(197,)

In [15]:
sql = """
SELECT *
FROM stocks
"""
stocks = pd.read_sql(sql, conlt)
stocks.shape

(230, 15)

In [17]:
sql = """
SELECT *
FROM tickers
"""
tickers = pd.read_sql(sql, conlt)
tickers.shape

(391, 9)

In [19]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
#df_percent.style.format(format_dict)
df_percent.shape

(5, 7)

In [21]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM qt_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 0


In [23]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
tickers.sample(5)

,name,id
298,SQ,472
177,LHSC,272
220,PLE,360
99,EA,148
199,MTI,314


In [25]:
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
#df_ins.style.format(format_dict)
df_ins.shape

(5, 8)

In [27]:
rcds = df_ins.values.tolist()
len(rcds)

5

In [29]:
sql = 'SELECT * FROM qt_profits'
df_qt_profits = pd.read_sql(sql, conlt)
before_insert = df_qt_profits.shape[0]

In [31]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO qt_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

In [33]:
sql = 'SELECT * FROM qt_profits'
df_qt_profits = pd.read_sql(sql, conlt)
after_insert = df_qt_profits.shape[0]

In [35]:
insert_records = after_insert - before_insert

In [37]:
print(f"({insert_records}) records inserted")

(5) records inserted


### End of loop

In [40]:
criteria_1 = df_ins.q_amt_c > 440_000
#df_ins.loc[criteria_1, cols].style.format(format_dict)
df_ins.shape

(5, 8)

In [42]:
criteria_2 = df_ins.q_amt_p > 400_000
#df_ins.loc[criteria_2, cols].style.format(format_dict)
df_ins.shape

(5, 8)

In [44]:
criteria_3 = df_ins.percent > 10.00
#df_ins.loc[criteria_3, cols].style.format(format_dict)
df_ins.shape

(5, 8)

In [46]:
df_ins_criteria = criteria_1 & criteria_2 & criteria_3
#df_ins.loc[df_ins_criteria, cols].style.format(format_dict)
df_ins.shape

(5, 8)

In [48]:
df_ins[df_ins_criteria].sort_values(by=["percent"], ascending=[False]).head(5).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id


In [50]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head(5).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id


In [52]:
df_ins[df_ins_criteria].sort_values(by=["name"], ascending=[True]).head(5).style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent,id


In [54]:
sql = """
SELECT name, COUNT(*) AS 'Nbr. of Group of 4 Quarters'
FROM qt_profits
GROUP BY name
HAVING COUNT(*) = 2
ORDER BY COUNT(*) DESC, name
"""
print(sql)

tmp = pd.read_sql(sql, conlt)
tmp


SELECT name, COUNT(*) AS 'Nbr. of Group of 4 Quarters'
FROM qt_profits
GROUP BY name
HAVING COUNT(*) = 2
ORDER BY COUNT(*) DESC, name



,name,Nbr. of Group of 4 Quarters
0,BAT-3K,2


In [56]:
conlt.commit()
conlt.close()

In [58]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:08:22 15:59:23
